In [24]:
import os
import warnings
import pickle
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")

In [25]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

In [26]:
DATA_PATH       = "/kaggle/input/datasets/quanghuylt/combined/combined.csv"   
METRICS_DIR     = "/kaggle/working/reports/metrics/arimax/"  # save metrics of a single ticker
RESULTS_DIR     = "/kaggle/working/data/meta_test/"  # save final arimax predicted results on the test set, used to test the meta-learner
FIGURES_DIR     = "/kaggle/working/reports/figures/"   
MODEL_DIR       = "/kaggle/working/models/arimax/"   
OOF_DIR         = "/kaggle/working/data/meta_train/"  # save arimax predicted results to train meta-learner
SUMMARY_METRICS = "/kaggle/working/reports/metrics/arimax/arimax_all_tickers.csv"  # save metrics of all tickers when refitted on full training set

SYMBOLS   = ["VNM", "FPT", "MSN", "VCB", "VIC", "HPG"]
EXOG_COLS = ["dxy", "fed_rate", "gold", "oil", "sp500"]  # macro exogenous variables

# fix 5
TRAIN_RATIO = 0.8
N_FOLDS     = 10 
GAP_DAYS    = 10   # purge gap between train end and val start to avoid leakage
# more folds (less data is used in the first train oof) 
# less gap days to avoid nan input data for meta lear

In [27]:
def split_data(df: pd.DataFrame):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    split_idx = int(len(df) * TRAIN_RATIO)
    train_df  = df.iloc[:split_idx].reset_index(drop=True)
    test_df   = df.iloc[split_idx:].reset_index(drop=True)
    return train_df, test_df

In [28]:
# fix 2
def scale_data(train_df, test_df):
    close_scaler = MinMaxScaler()
    exog_scaler  = MinMaxScaler()

    train_df["Close"]       = close_scaler.fit_transform(train_df[["Close"]])
    train_df[EXOG_COLS]     = exog_scaler.fit_transform(train_df[EXOG_COLS])

    test_df["Close"]        = close_scaler.transform(test_df[["Close"]])
    test_df[EXOG_COLS]      = exog_scaler.transform(test_df[EXOG_COLS])

    return train_df, test_df, close_scaler

In [29]:
# fix 4
def prepare_exog(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df[EXOG_COLS] = df[EXOG_COLS].diff()
    df = df.dropna().reset_index(drop=True)
    return df

In [39]:
import shutil
from IPython.display import FileLink

shutil.make_archive('/kaggle/working/data_download', 'zip', '/kaggle/working/data')
FileLink(r'data_download.zip')

shutil.make_archive('/kaggle/working/reports_download', 'zip', '/kaggle/working/reports')
FileLink(r'reports_download.zip')


shutil.make_archive('/kaggle/working/models_download', 'zip', '/kaggle/working/models')
FileLink(r'models_download.zip')

/kaggle/working/models_download.zip

In [40]:
shutil.make_archive('/kaggle/working/data_download', 'zip', '/kaggle/working/data')
FileLink(r'data_download.zip')


/kaggle/working/data_download.zip

In [41]:
shutil.make_archive('/kaggle/working/reports_download', 'zip', '/kaggle/working/reports')
FileLink(r'reports_download.zip')

/kaggle/working/reports_download.zip

In [31]:
def find_best_order(endog: pd.Series, exog: pd.DataFrame) -> tuple:
    log.info("    Running auto_arima to find best (p,d,q)...")
    model = auto_arima(
        endog,
        exogenous=exog,
        seasonal=False,          
        test="adf", 
        d=None,
        stepwise=True,
        information_criterion="aic",
        error_action="ignore",
        suppress_warnings=True,
        trace=False,
        max_p=3, max_q=3,        # cap order to avoid overfitting
    )
    order = model.order
    log.info(f"Best order={order}")
    return order

In [32]:
def run_oof(train_df: pd.DataFrame, order: tuple, ticker: str) -> pd.DataFrame:
    n         = len(train_df)
    fold_size = n // (N_FOLDS + 1)
    oof_preds = np.full(n, np.nan)  # nan everywhere, only val windows will be filled

    log.info(f"    OOF: {N_FOLDS} folds, fold_size={fold_size}, gap={GAP_DAYS}")

    for k in range(1, N_FOLDS + 1):
        train_end = k * fold_size
        val_start = train_end + GAP_DAYS   # skip gap days to prevent leakage through rolling features
        # val_end   = val_start + fold_size
        val_end = (k + 1) * fold_size if k < N_FOLDS else n

        if val_end > n:
            break

        fold_train = train_df.iloc[:train_end]
        fold_val   = train_df.iloc[val_start:val_end]

        try:
            result = ARIMA(
                fold_train["Close"],
                exog=fold_train[EXOG_COLS],   # macro variables as exogenous regressors
                order=order,
            ).fit()

            # forecast = result.forecast(
            #     steps=len(fold_val),
            #     exog=fold_val[EXOG_COLS],     # must pass future exog values when forecasting
            # )

            # fix 3
            # forecast 
            result_applied = result.apply(
                fold_val["Close"],
                exog=fold_val[EXOG_COLS],
                refit=False,
            )
            
            oof_preds[val_start:val_end] = result_applied.fittedvalues.values

            mae = np.mean(np.abs(result_applied.fittedvalues.values - fold_val["Close"].values))
            log.info(f"      Fold {k}: train=[0:{train_end}]  "
                     f"val=[{val_start}:{val_end}]  MAE={mae:.4f}")

        except Exception as e:
            log.warning(f"      Fold {k} failed: {e}")
            oof_preds[val_start:val_end] = np.nan

        # calculate nan count due to oof training
    nan_count = np.isnan(oof_preds).sum()
    print(f'  oof nan: {nan_count}/{n} ({nan_count/n:.1%}) — only the first ~{fold_size + GAP_DAYS} rows are uncovered')

    return pd.DataFrame({
        "Ticker":                 ticker,
        "Date":                   train_df["Date"].values,
        "Close":                  train_df["Close"].values,
        "ARIMAX_Predicted_Close": oof_preds,   # nan at gap regions between folds
    })

In [33]:
def predict_test(final_result, test_df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    # forecast = final_result.forecast(
    #     steps=len(test_df),
    #     exog=test_df[EXOG_COLS],
    # )
# fix 1
    
    # use the model to predict the final test set without refitting the parameters
    result_applied = final_result.apply(
        test_df["Close"],
        exog=test_df[EXOG_COLS],
        refit=False
    )

    return pd.DataFrame({
        "Ticker":                 ticker,
        "Date":                   test_df["Date"].values,
        "Close":                  test_df["Close"].values,
        "ARIMAX_Predicted_Close": result_applied.fittedvalues.values,
    })

In [34]:
def calc_metrics(
    df: pd.DataFrame,
    actual_col: str = "Close",
    pred_col:   str = "ARIMAX_Predicted_Close",
    ticker_col: str = "Ticker",
) -> pd.DataFrame:

    # recurse per ticker when df contains multiple tickers
    if ticker_col and ticker_col in df.columns and df[ticker_col].nunique() > 1:
        return (
            df.groupby(ticker_col)
              .apply(lambda x: calc_metrics(x, actual_col, pred_col, ticker_col=None))
              .reset_index()
        )

    df = df.copy()
    df = df.dropna(subset=[actual_col, pred_col])   # drop nan rows from oof gap regions
    df["Prev_Actual"] = df[actual_col].shift(1)
    df["Prev_Pred"]   = df[pred_col].shift(1)
    df = df.dropna(subset=["Prev_Actual", "Prev_Pred"])

    y_true = df[actual_col]
    y_pred = df[pred_col]

    mse  = mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_true - y_pred) / y_true + 1e-8))
    r2   = r2_score(y_true, y_pred)

    actual_dir = np.sign(y_true.values - df["Prev_Actual"].values)  # +1 up, -1 down, 0 flat
    pred_dir   = np.sign(y_pred.values - df["Prev_Pred"].values)
    da         = np.mean(actual_dir == pred_dir)                     # directional accuracy

    mask = actual_dir != 0                                            # exclude flat days from tpa
    tpa  = np.mean(actual_dir[mask] == pred_dir[mask]) if np.any(mask) else np.nan

    # volatility rmse: compare de-meaned series so level bias doesn't inflate the error
    actual_vol = y_true - y_true.mean()
    pred_vol   = y_pred - y_pred.mean()
    v_rmse     = np.sqrt(mean_squared_error(actual_vol, pred_vol))

    return pd.DataFrame([{
        "MSE":    mse,
        "MAE":    mae,
        "MAPE":   mape,
        "RMSE":   rmse,
        "R2":     r2,
        "DA":     da,
        "TPA":    tpa,
        "V-RMSE": v_rmse,
    }])

In [35]:
def plot_results(df: pd.DataFrame, save_dir: str) -> None:
    ticker  = df["Ticker"].iloc[0]
    df_plot = df.dropna(subset=["ARIMAX_Predicted_Close"])

    os.makedirs(save_dir, exist_ok=True)
    sns.set(style="whitegrid")

    fig, axes = plt.subplots(2, 1, figsize=(14, 10))

    # top panel: actual vs predicted close price
    axes[0].plot(df_plot["Date"], df_plot["Close"],
                 label="Actual Close", color="#1f77b4", linewidth=2)
    axes[0].plot(df_plot["Date"], df_plot["ARIMAX_Predicted_Close"],
                 label="ARIMAX Predicted", color="#d62728", linestyle="--", linewidth=1.5)
    axes[0].set_title(f"ARIMAX predicted results: {ticker}", fontsize=15)
    axes[0].set_xlabel("Date", fontsize=12)
    axes[0].set_ylabel("Price", fontsize=12)
    axes[0].legend(loc="best")
    axes[0].grid(True, alpha=0.3)
    axes[0].tick_params(axis="x", rotation=30)

    # bottom panel: residual error per day
    error  = df_plot["Close"] - df_plot["ARIMAX_Predicted_Close"]
    colors = np.where(error >= 0, "#2ca02c", "#d62728")   # green = under-predicted, red = over-predicted
    axes[1].bar(df_plot["Date"], error, color=colors, alpha=0.6, width=1.5)
    axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
    axes[1].set_title(f"Prediction error: {ticker}", fontsize=13)
    axes[1].set_xlabel("Date", fontsize=12)
    axes[1].set_ylabel("Error (Actual - Predicted)", fontsize=12)
    axes[1].grid(True, alpha=0.3)
    axes[1].tick_params(axis="x", rotation=30)

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"{ticker}.png")
    plt.savefig(save_path, bbox_inches="tight", dpi=150)
    plt.close()
    log.info(f"    Plot saved: {save_path}")

In [37]:
def train_one_ticker(ticker: str) -> pd.DataFrame:
    log.info(f"========== {ticker} ==========")

    df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
    df = df[df["Ticker"] == ticker].sort_values("Date").reset_index(drop=True)

    df = prepare_exog(df)

    train_df, test_df = split_data(df)

    train_df, test_df, close_scaler = scale_data(train_df, test_df)
    
    train_df = prepare_exog(train_df)
    test_df = prepare_exog(test_df)

    log.info(f"  Train: {len(train_df)} rows  Test: {len(test_df)} rows")

    order = find_best_order(train_df["Close"], train_df[EXOG_COLS])
    oof_df = run_oof(train_df, order, ticker)

    # refit on full train set to use all available signal before predicting test
    log.info("    Refitting on full train set...")
    final_result = ARIMA(
        train_df["Close"],
        exog=train_df[EXOG_COLS],
        order=order,
    ).fit()

    test_result_df = predict_test(final_result, test_df, ticker)

    mask_oof = oof_df["ARIMAX_Predicted_Close"].notna()
    oof_df.loc[mask_oof, ["ARIMAX_Predicted_Close"]] = close_scaler.inverse_transform(oof_df.loc[mask_oof, ["ARIMAX_Predicted_Close"]])
    oof_df[["Close"]] = close_scaler.inverse_transform(oof_df[["Close"]])

    # inverse minmax scaler to get the original values
    test_result_df[["ARIMAX_Predicted_Close"]] = close_scaler.inverse_transform(test_result_df[["ARIMAX_Predicted_Close"]])
    test_result_df[["Close"]] = close_scaler.inverse_transform(test_result_df[["Close"]])


    metrics_df = calc_metrics(oof_df)
    metrics_df.insert(0, "Ticker", ticker)

    log.info(f"  OOF  — MAE={metrics_df['MAE'].iloc[0]:.2f}"
             f"DA={metrics_df['DA'].iloc[0]:.2%}  "
             f"R2={metrics_df['R2'].iloc[0]:.4f}")

    test_metrics = calc_metrics(test_result_df)
    log.info(f"  Test — MAE={test_metrics['MAE'].iloc[0]:.2f}"
             f"DA={test_metrics['DA'].iloc[0]:.2%}  "
             f"R2={test_metrics['R2'].iloc[0]:.4f}")

    os.makedirs(METRICS_DIR, exist_ok=True)
    metrics_df.to_csv(
        os.path.join(METRICS_DIR, f"ARIMAX_{ticker}_metrics.csv"), index=False
    )

    # oof predictions saved for downstream ridge meta-learner stacking
    os.makedirs(OOF_DIR, exist_ok=True)
    oof_df.to_csv(
        os.path.join(OOF_DIR, f"oof_arimax_{ticker}.csv"), index=False
    )

    os.makedirs(RESULTS_DIR, exist_ok=True)
    test_result_df.to_csv(
        os.path.join(RESULTS_DIR, f"ARIMAX_{ticker}_predictions.csv"), index=False
    )

    plot_results(test_result_df, save_dir=FIGURES_DIR)

    # (Phần lưu pickle model và return mình giữ nguyên như code cũ của bạn)
    os.makedirs(MODEL_DIR, exist_ok=True)
    with open(os.path.join(MODEL_DIR, f"arimax_{ticker}.pkl"), "wb") as f:
        pickle.dump({"result": final_result, "order": order, "ticker": ticker}, f)

    log.info(f"  All outputs saved for {ticker}")
    return metrics_df

In [38]:
all_metrics = []

for ticker in SYMBOLS:
    try:
        m = train_one_ticker(ticker)
        all_metrics.append(m)
    except Exception as e:
        log.error(f"{ticker} FAILED: {e}")

if all_metrics:
    summary_df = pd.concat(all_metrics, ignore_index=True)
    os.makedirs(os.path.dirname(SUMMARY_METRICS) or ".", exist_ok=True)
    summary_df.to_csv(SUMMARY_METRICS, index=False)
    log.info(f"\nSummary saved: {SUMMARY_METRICS}")
    print("\n" + summary_df.to_string(index=False))

log.info("Done.")

04:01:52  INFO      ========== VNM ==========
04:01:52  INFO        Train: 3257 rows  Test: 814 rows
04:01:52  INFO          Running auto_arima to find best (p,d,q)...
04:01:56  INFO      Best order=(0, 1, 0)
04:01:56  INFO          OOF: 10 folds, fold_size=296, gap=10
04:01:56  INFO            Fold 1: train=[0:296]  val=[306:592]  MAE=0.0015
04:01:57  INFO            Fold 2: train=[0:592]  val=[602:888]  MAE=0.0031
04:01:57  INFO            Fold 3: train=[0:888]  val=[898:1184]  MAE=0.0038
04:01:58  INFO            Fold 4: train=[0:1184]  val=[1194:1480]  MAE=0.0047
04:01:59  INFO            Fold 5: train=[0:1480]  val=[1490:1776]  MAE=0.0075
04:01:59  INFO            Fold 6: train=[0:1776]  val=[1786:2072]  MAE=0.0103
04:02:00  INFO            Fold 7: train=[0:2072]  val=[2082:2368]  MAE=0.0116
04:02:01  INFO            Fold 8: train=[0:2368]  val=[2378:2664]  MAE=0.0117
04:02:02  INFO            Fold 9: train=[0:2664]  val=[2674:2960]  MAE=0.0103
04:02:04  INFO            Fold 10: t

  oof nan: 396/3257 (12.2%) — only the first ~306 rows are uncovered


04:02:05  INFO        OOF  — MAE=0.71DA=41.68%  R2=0.9840
04:02:05  INFO        Test — MAE=0.61DA=43.91%  R2=0.9475
04:02:06  INFO          Plot saved: /kaggle/working/reports/figures/VNM.png
04:02:06  INFO        All outputs saved for VNM
04:02:06  INFO      ========== FPT ==========
04:02:07  INFO        Train: 3257 rows  Test: 814 rows
04:02:07  INFO          Running auto_arima to find best (p,d,q)...
04:02:13  INFO      Best order=(0, 1, 3)
04:02:13  INFO          OOF: 10 folds, fold_size=296, gap=10
04:02:14  INFO            Fold 1: train=[0:296]  val=[306:592]  MAE=0.0012
04:02:14  INFO            Fold 2: train=[0:592]  val=[602:888]  MAE=0.0009
04:02:15  INFO            Fold 3: train=[0:888]  val=[898:1184]  MAE=0.0017
04:02:16  INFO            Fold 4: train=[0:1184]  val=[1194:1480]  MAE=0.0016
04:02:18  INFO            Fold 5: train=[0:1480]  val=[1490:1776]  MAE=0.0016
04:02:19  INFO            Fold 6: train=[0:1776]  val=[1786:2072]  MAE=0.0031
04:02:21  INFO            Fold

  oof nan: 396/3257 (12.2%) — only the first ~306 rows are uncovered


04:02:28  INFO        OOF  — MAE=0.24DA=46.40%  R2=0.9946
04:02:28  INFO        Test — MAE=1.09DA=49.57%  R2=0.9954
04:02:30  INFO          Plot saved: /kaggle/working/reports/figures/FPT.png
04:02:30  INFO        All outputs saved for FPT
04:02:30  INFO      ========== MSN ==========
04:02:30  INFO        Train: 3257 rows  Test: 814 rows
04:02:30  INFO          Running auto_arima to find best (p,d,q)...
04:02:58  INFO      Best order=(2, 1, 2)
04:02:58  INFO          OOF: 10 folds, fold_size=296, gap=10
04:02:59  INFO            Fold 1: train=[0:296]  val=[306:592]  MAE=0.0106
04:03:00  INFO            Fold 2: train=[0:592]  val=[602:888]  MAE=0.0073
04:03:01  INFO            Fold 3: train=[0:888]  val=[898:1184]  MAE=0.0054
04:03:03  INFO            Fold 4: train=[0:1184]  val=[1194:1480]  MAE=0.0048
04:03:05  INFO            Fold 5: train=[0:1480]  val=[1490:1776]  MAE=0.0039
04:03:08  INFO            Fold 6: train=[0:1776]  val=[1786:2072]  MAE=0.0079
04:03:12  INFO            Fold

...

  oof nan: 396/3257 (12.2%) — only the first ~306 rows are uncovered


04:04:55  INFO        OOF  — MAE=0.38DA=42.45%  R2=0.9891
04:04:55  INFO        Test — MAE=1.14DA=49.45%  R2=0.9972
04:04:57  INFO          Plot saved: /kaggle/working/reports/figures/VIC.png
04:04:57  INFO        All outputs saved for VIC
04:04:57  INFO      ========== HPG ==========
04:04:57  INFO        Train: 3257 rows  Test: 814 rows
04:04:57  INFO          Running auto_arima to find best (p,d,q)...
04:05:19  INFO      Best order=(2, 1, 2)
04:05:19  INFO          OOF: 10 folds, fold_size=296, gap=10
04:05:20  INFO            Fold 1: train=[0:296]  val=[306:592]  MAE=0.0007
04:05:21  INFO            Fold 2: train=[0:592]  val=[602:888]  MAE=0.0007
04:05:21  INFO            Fold 3: train=[0:888]  val=[898:1184]  MAE=0.0013
04:05:23  INFO            Fold 4: train=[0:1184]  val=[1194:1480]  MAE=0.0017
04:05:24  INFO            Fold 5: train=[0:1480]  val=[1490:1776]  MAE=0.0020
04:05:25  INFO            Fold 6: train=[0:1776]  val=[1786:2072]  MAE=0.0043
04:05:26  INFO            Fold

  oof nan: 396/3257 (12.2%) — only the first ~306 rows are uncovered


04:05:34  INFO        OOF  — MAE=0.16DA=45.49%  R2=0.9930
04:05:34  INFO        Test — MAE=0.28DA=46.25%  R2=0.9873
04:05:36  INFO          Plot saved: /kaggle/working/reports/figures/HPG.png
04:05:36  INFO        All outputs saved for HPG
04:05:36  INFO      
Summary saved: /kaggle/working/reports/metrics/arimax/arimax_all_tickers.csv
04:05:36  INFO      Done.



Ticker      MSE      MAE     MAPE     RMSE       R2       DA      TPA   V-RMSE
   VNM 9.641187 0.713652 0.013383 3.105026 0.983954 0.416783 0.502741 3.100189
   FPT 1.170340 0.239494 0.013535 1.081822 0.994621 0.463986 0.518765 1.080456
   MSN 9.526597 1.106050 0.017642 3.086519 0.983737 0.446503 0.528778 3.083025
   VCB 1.612614 0.337120 0.016023 1.269888 0.992121 0.456993 0.506393 1.268127
   VIC 3.204067 0.380650 0.014881 1.789991 0.989134 0.424476 0.512453 1.788248
   HPG 0.425248 0.155173 0.018850 0.652110 0.993002 0.454895 0.514026 0.651477
